# Visualization Orchestrator

Interact with the hierarchical, capability-based orchestrator (`visualization_orchestrator/`) through one object, `VisualizationOrchestrator` -- it wraps the VTK session, the camera agent, the isovalue agent, and the planner/executor/verifier stack.

Just instantiate it once, then call `.run("...")` with any instruction, in any order, as many times as you like -- state carries forward between calls like a multi-turn conversation. No need to touch the registry/planner/executor plumbing directly.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from IPython.display import Image, display

from visualization_orchestrator import VisualizationOrchestrator


def show():
    """Display the orchestrator's most recently rendered image."""
    display(Image(filename=orchestrator.image_path))

In [ ]:
# Requires OPENAI_API_KEY set (see ../.env.example) -- the planner, both specialists,
# and the final verifier all make real LLM calls.
orchestrator = VisualizationOrchestrator(
    dataset_path="../data/skull_256x256x256_uint8.raw",
    dimensions=(256, 256, 256),
    isovalue=40,
    output_dir="../output/orchestrator_notebook",
)
show()

## Give it an instruction

Run any of the cells below in any order, or edit the instruction text and re-run. Each call plans a small task graph, runs whichever specialist(s) the plan requires (camera, isovalue, or both -- never a fixed pipeline), and verifies the result against the instruction.

In [ ]:
result = orchestrator.run("Show the inside of the head from behind.")
show()

In [ ]:
result = orchestrator.run("Show the skull from the left.")
show()

In [ ]:
result = orchestrator.run("Reduce the surface noise.")
show()

In [ ]:
result = orchestrator.run("Rotate to the right.")
show()

In [ ]:
result = orchestrator.run("Move to the posterior view first, then find the best isovalue for showing the inside.")
show()

## Try your own

In [ ]:
result = orchestrator.run("")
show()

## Inspect the last result

`result` (from whichever `.run()` cell you ran most recently) is an `ExecutionResult`: the interpreted plan's outcome, per-task records, replan/agent-call counts, and the final verification.

In [ ]:
print(f"success={result.success}  replans_used={result.replans_used}  total_agent_calls={result.total_agent_calls}")
print()
for record in result.task_records:
    print(f"- {record.task_id}: agent={record.agent_id} status={record.status}")
    print(f"    reason: {record.reason}")
print()
if result.final_verification:
    v = result.final_verification
    print(f"final verification: success={v.success} confidence={v.confidence}")
    print(f"  diagnosis: {v.diagnosis}")
    print(f"  satisfied: {v.satisfied_criteria}")
    print(f"  unsatisfied: {v.unsatisfied_criteria}")

In [ ]:
# Full history of every .run() call this session, in order.
for i, r in enumerate(orchestrator.history):
    print(f"{i}: success={r.success} tasks={[t.task_id for t in r.task_records]}")